In [1]:
# To nowy Spark bo jest SQL
from pyspark.sql import SparkSession

In [2]:
# PRACA DOMOWA
# Jak zrobić transpozycję tabelki Sparkowej? - poszukać informacji
# Nie chodzi o rozwiązanie z to_pandas
# Będzie z SparkContext

In [3]:
# Minimalnistyczne uruchomienie Sparka
# spark = SparkSession.builder.getOrCreate()

In [4]:
spark = (
    SparkSession.builder
    .appName("Lab2-Transactions")
    .getOrCreate()
)

In [5]:
spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [6]:
df = spark.read.json("transactions_10k.jsonl")

print(f"Liczba rekordów: {df.count()}")
df.printSchema()

Liczba rekordów: 10000
root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [7]:
# Zmiana zmiennej timestamp na typ timestamp
from pyspark.sql.functions import to_timestamp, col

df = df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

df.printSchema()  # timestamp powinien być teraz 'timestamp (nullable = true)'

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [8]:
from pyspark.sql.functions import count, sum as _sum, avg, round as _round

store_summary = (
    df.groupBy("store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
        _round(avg("amount"), 2).alias("srednia_PLN"),
    )
    .orderBy("store")
)

In [9]:
store_summary.show()

+--------+---------+----------+-----------+
|   store|liczba_tx|  suma_PLN|srednia_PLN|
+--------+---------+----------+-----------+
|  Gdańsk|     2498|1021266.35|     408.83|
|  Kraków|     2522|1025896.95|     406.78|
|Warszawa|     2424| 961642.24|     396.72|
| Wrocław|     2556|1002739.21|     392.31|
+--------+---------+----------+-----------+



In [10]:
from pyspark.sql.functions import min as _min, max as _max

# Policz sumę, minimum i maksimum kwoty dla każdej kategorii

# TWÓJ KOD
category_summary = (
    df.groupBy("category")
    .agg(
        _round(_sum('amount'), 2).alias('suma_PLN'),
        _round(_min('amount'), 2).alias('min_transaction'),
        _round(_max('amount'), 2).alias('max_transaction'),
    )
    .orderBy("category")
)

In [11]:
category_summary.show()

+-----------+----------+---------------+---------------+
|   category|  suma_PLN|min_transaction|max_transaction|
+-----------+----------+---------------+---------------+
|elektronika|1520770.69|            9.0|         9999.0|
|    książki| 851382.08|            5.0|        9107.25|
|     odzież| 849877.55|            5.0|        9696.63|
|    żywność| 789514.43|            5.0|        6916.92|
+-----------+----------+---------------+---------------+



In [12]:
from pyspark.sql.functions import window

In [13]:
hourly = (
    df.groupBy(window("timestamp", "1 hour"))    # okno 1-godzinne
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .orderBy("window")
)
hourly.show(truncate=False)

+------------------------------------------+---------+----------+
|window                                    |liczba_tx|suma_PLN  |
+------------------------------------------+---------+----------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|3150     |1241911.3 |
|{2026-04-12 09:00:00, 2026-04-12 10:00:00}|4661     |1896230.21|
|{2026-04-12 10:00:00, 2026-04-12 11:00:00}|2189     |873403.24 |
+------------------------------------------+---------+----------+



In [14]:
hourly.printSchema()

root
 |-- window: struct (nullable = false)
 |    |-- start: timestamp (nullable = true)
 |    |-- end: timestamp (nullable = true)
 |-- liczba_tx: long (nullable = false)
 |-- suma_PLN: double (nullable = true)



In [15]:
(
    hourly
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .show(truncate=False)
)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
+-------------------+-------------------+---------+----------+



In [16]:
df.printSchema()

root
 |-- amount: double (nullable = true)
 |-- category: string (nullable = true)
 |-- store: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)



In [17]:
# Policz transakcje i sumę per sklep w każdym 30-minutowym oknie. 
# Posortuj po oknie, a w ramach okna po sklepie

# TWÓJ KOD
tumbling_30_min = (
    df.groupBy(window("timestamp", "30 minutes"), "store")
    .agg(
        count('tx_id').alias('liczba_tx'),
        _round(_sum('amount'), 2).alias('suma_PLN'),
    )
    .orderBy('window', 'store')
)

tumbling_30_min.show(truncate=False)

+------------------------------------------+--------+---------+---------+
|window                                    |store   |liczba_tx|suma_PLN |
+------------------------------------------+--------+---------+---------+
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Gdańsk  |252      |93391.22 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Kraków  |289      |117786.42|
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Warszawa|275      |88441.58 |
|{2026-04-12 08:00:00, 2026-04-12 08:30:00}|Wrocław |296      |111540.59|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Gdańsk  |514      |209187.85|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Kraków  |532      |223541.41|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Warszawa|490      |182435.06|
|{2026-04-12 08:30:00, 2026-04-12 09:00:00}|Wrocław |502      |215587.17|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Gdańsk  |619      |253364.95|
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|Kraków  |590      |224358.03|
|{2026-04-12 09:00:00, 2026-04-12 09:3

In [18]:
from pyspark.sql.functions import desc

# W której godzinie sklep “Kraków” miał najwyższy przychód?
# Filtruj najpierw po sklepie, potem zrób okno godzinne, posortuj malejąco po sumie
# TWÓJ KOD
(
    df.where("store = 'Kraków'")
    .groupBy(window('timestamp', '1 hour'))
    .agg(
        _round(_sum('amount'), 2).alias('suma_PLN')
    )
    .select(
        col('window.start').alias('od'),
        col('window.end').alias('do'),
        'suma_PLN'
    )
    .orderBy(desc('suma_PLN'))
    .show(truncate=False)
)

+-------------------+-------------------+---------+
|od                 |do                 |suma_PLN |
+-------------------+-------------------+---------+
|2026-04-12 09:00:00|2026-04-12 10:00:00|483309.86|
|2026-04-12 08:00:00|2026-04-12 09:00:00|341327.83|
|2026-04-12 10:00:00|2026-04-12 11:00:00|201259.26|
+-------------------+-------------------+---------+



In [19]:
sliding = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))  # szerokość 1h, krok 30min
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
    .select(
        col("window.start").alias("od"),
        col("window.end").alias("do"),
        "liczba_tx",
        "suma_PLN",
    )
    .orderBy("od")
)
sliding.show(truncate=False)

+-------------------+-------------------+---------+----------+
|od                 |do                 |liczba_tx|suma_PLN  |
+-------------------+-------------------+---------+----------+
|2026-04-12 07:30:00|2026-04-12 08:30:00|1112     |411159.81 |
|2026-04-12 08:00:00|2026-04-12 09:00:00|3150     |1241911.3 |
|2026-04-12 08:30:00|2026-04-12 09:30:00|4443     |1753033.6 |
|2026-04-12 09:00:00|2026-04-12 10:00:00|4661     |1896230.21|
|2026-04-12 09:30:00|2026-04-12 10:30:00|3696     |1557641.39|
|2026-04-12 10:00:00|2026-04-12 11:00:00|2189     |873403.24 |
|2026-04-12 10:30:00|2026-04-12 11:30:00|749      |289709.95 |
+-------------------+-------------------+---------+----------+



In [20]:
tumbling_rows = (
    df.groupBy(window("timestamp", "1 hour"))
    .agg(count("tx_id"))
    .count()
)
sliding_rows = (
    df.groupBy(window("timestamp", "1 hour", "30 minutes"))
    .agg(count("tx_id"))
    .count()
)
print(f"Tumbling (1h):          {tumbling_rows} okien")
print(f"Sliding  (1h / 30min):  {sliding_rows} okien")

# Odpowiedz w komentarzu: dlaczego sliding ma więcej wierszy?
# TWOJA ODPOWIEDŹ:
# Ponieważ krok 30 min jest zarówno do przodu jak i do tyłu

Tumbling (1h):          3 okien
Sliding  (1h / 30min):  7 okien


In [21]:
# Odpowiedz na pytania w komentarzach:

# 1. Ile transakcji jest w oknie 09:00–10:00?
#    Sprawdź w wyniku zadania 3.1.
#    ODPOWIEDŹ: 4661

# 2. Jaka jest różnica między groupBy("store") a groupBy(window(...), "store")?
#    ODPOWIEDŹ: W pierwszej opcji bierzemy pod uwagę cały przedział czasu,
#               a w drugiej dla każdego okna czasowego robimy statystyki dla każdego sklepu

# 3. W oknie sliding 1h/30min — ile okien zawiera transakcje z godziny 09:30?
#    Wskazówka: narysuj oś czasu.
#    ODPOWIEDŹ: 3 okna - 8:30 - 9:30, 9:00 - 10:00, 9:30 - 10:30

In [22]:
# 1. Znajdź godzinę, w której sklep Gdańsk miał najniższą średnią kwotę transakcji.
# 2. Policz ile transakcji per kategoria było w oknie 09:00–09:30.
# 3. Zrób okno 15-minutowe i sprawdź w której ćwierćgodzinie był szczyt transakcji (łącznie dla wszystkich sklepów).

In [23]:
# 1.
(
    df.where("store = 'Gdańsk'")
    .groupBy(window('timestamp', '1 hour'))
    .agg(
        avg('amount').alias('średnia_PLN')
    )
    .orderBy('średnia_PLN')
    .limit(1)
    .show(truncate=False)
)

+------------------------------------------+-----------------+
|window                                    |średnia_PLN      |
+------------------------------------------+-----------------+
|{2026-04-12 08:00:00, 2026-04-12 09:00:00}|395.0118407310706|
+------------------------------------------+-----------------+



In [31]:
# 2.
(
    df.groupBy(window('timestamp', '30 minutes'), 'category')
    .agg(
        count('tx_id').alias('liczba_tx')
    )
    .where("to_char(window.start, 'yyyy-MM-dd HH:mm:ss') like '%9:00%'")
    .orderBy('window', 'category')
    .show(truncate=False)
)

+------------------------------------------+-----------+---------+
|window                                    |category   |liczba_tx|
+------------------------------------------+-----------+---------+
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|elektronika|611      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|książki    |622      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|odzież     |605      |
|{2026-04-12 09:00:00, 2026-04-12 09:30:00}|żywność    |567      |
+------------------------------------------+-----------+---------+



In [34]:
# 3.
(
    df.groupBy(window('timestamp', '15 minutes'))
    .agg(_round(_sum('amount'), 2).alias('suma_PLN'))
    .orderBy(desc('suma_PLN'))
    .show(truncate=False)
)

+------------------------------------------+---------+
|window                                    |suma_PLN |
+------------------------------------------+---------+
|{2026-04-12 09:30:00, 2026-04-12 09:45:00}|504943.74|
|{2026-04-12 09:15:00, 2026-04-12 09:30:00}|481566.97|
|{2026-04-12 08:45:00, 2026-04-12 09:00:00}|475251.18|
|{2026-04-12 09:45:00, 2026-04-12 10:00:00}|469004.36|
|{2026-04-12 09:00:00, 2026-04-12 09:15:00}|440715.14|
|{2026-04-12 10:00:00, 2026-04-12 10:15:00}|359254.89|
|{2026-04-12 08:30:00, 2026-04-12 08:45:00}|355500.31|
|{2026-04-12 10:15:00, 2026-04-12 10:30:00}|224438.4 |
|{2026-04-12 08:15:00, 2026-04-12 08:30:00}|213061.19|
|{2026-04-12 08:00:00, 2026-04-12 08:15:00}|198098.62|
|{2026-04-12 10:30:00, 2026-04-12 10:45:00}|156900.51|
|{2026-04-12 10:45:00, 2026-04-12 11:00:00}|132809.44|
+------------------------------------------+---------+



In [ ]:
spark.stop()